# Projekt 02 (medium): Das ehrliche Modellrennen — Pipelines, CV & Tuning

**Ziel:** Mehrere Modellfamilien fair per Pipeline + Kreuzvalidierung vergleichen,
die zwei besten tunen, und **einmal ganz am Ende** ehrlich auf dem Testsatz messen.

**Daten:** Breast Cancer Wisconsin (Diagnostic), eingebaut in scikit-learn — 569 Proben,
30 numerische Features aus Zellkernbildern, Ziel: *malignant* (0) vs. *benign* (1).

Vorwissen: Skript Abschnitte 1.4-1.5 und 2.1-2.5. Weniger Anleitung als Projekt 01 —
die Konstruktion der Pipelines/des Grids sollst du selbst aus dem Skript ableiten.

## 1. Daten laden, explorieren, splitten

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

daten = load_breast_cancer()
X = pd.DataFrame(daten.data, columns=daten.feature_names)
y = daten.target  # 0 = malignant (boesartig), 1 = benign (gutartig)

print(X.shape)
print(pd.Series(y).map({0: "malignant", 1: "benign"}).value_counts(normalize=True))

**Aufgabe:** Mach den Split — stratifiziert (Skript 1.4: Klassenanteile in jedem
Split gleich halten), `test_size=0.2`, `random_state=42`. Der Testsatz wird ab jetzt
bis Schritt 6 **nicht** angefasst.

In [ ]:
# TODO: X_train, X_test, y_train, y_test = train_test_split(...)

print(f"Training: {len(X_train)}, Test: {len(X_test)}")
# Mini-Check: Klassenanteil im Test nah am Gesamtanteil (stratifiziert)
print(abs(y_test.mean() - y.mean()) < 0.02)

## 2. Baseline

Jedes ernsthafte Modell muss den `DummyClassifier` (immer die Mehrheitsklasse) schlagen.
Das ist die Nulllinie, gegen die du gleich alle CV-Ergebnisse einordnen kannst.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy Accuracy: {dummy.score(X_test, y_test):.3f}  (ROC-AUC waere 0.5 - Zufall)")

## 3. Pipelines fuer sechs Modellfamilien

**Aufgabe:** Baue ein Dictionary `modelle = {"Name": Pipeline(...), ...}` mit je einer
`Pipeline([("scaler", StandardScaler()), ("clf", <Modell>)])` fuer:

- `LogisticRegression` (`max_iter=5000`)
- `KNeighborsClassifier`
- `SVC` (RBF-Kernel ist Standard; `probability=True`, damit spaeter ROC-Kurven gehen)
- `DecisionTreeClassifier` (`random_state=42`)
- `RandomForestClassifier` (`random_state=42`)
- `GradientBoostingClassifier` (`random_state=42`)

Skript 2.4: Baeume/Ensembles brauchen keine Skalierung, aber die Pipeline schadet nicht
und haelt den Code einheitlich (eine Schleife fuer alle Modelle).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# TODO: modelle = {"Logistische Regression": Pipeline([...]), ...} fuer alle 6 Modelle

print(list(modelle.keys()))
print(len(modelle) == 6)

## 4. Fairer Vergleich per Kreuzvalidierung

**Aufgabe:** Fuer jedes Modell `cross_val_score` mit `StratifiedKFold(n_splits=5,
shuffle=True, random_state=42)` und `scoring="roc_auc"` auf **Trainingsdaten** berechnen.
Sammle die 5 Scores pro Modell (z. B. in `ergebnisse = {"Name": np.array([...]), ...}`)
und stelle sie als Boxplot dar.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TODO: ergebnisse = {}
# TODO: fuer jedes (name, pipeline) in modelle.items(): cross_val_score(...) berechnen und speichern

for name, scores in ergebnisse.items():
    print(f"{name:25s}  ROC-AUC = {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(ergebnisse.values(), tick_labels=ergebnisse.keys())
plt.xticks(rotation=30, ha="right")
plt.ylabel("ROC-AUC (5-fach CV)")
plt.title("Modellrennen: CV-ROC-AUC pro Modellfamilie")
plt.tight_layout()
plt.show()

**Aufgabe:** Notiere kurz (2-3 Saetze) in eigenen Worten: Welche 2 Modelle
gehen ins Tuning, und warum reicht der Mittelwert allein nicht — worauf achtest du bei
der Streuung (Boxplot-Breite)?

*(Deine Notiz hier ...)*

## 5. Hyperparameter-Tuning der Top 2

**Aufgabe:** Waehle deine zwei besten Modelle aus Schritt 4. Definiere fuer jedes ein
Parameter-Grid (Skript 2.5) und tune mit `GridSearchCV` (`cv=cv`, `scoring="roc_auc"`)
auf den **Trainingsdaten**. Beispiele fuer sinnvolle Grids:

- Random Forest: `clf__n_estimators`, `clf__max_depth`
- Gradient Boosting: `clf__n_estimators`, `clf__learning_rate`, `clf__max_depth`
- SVM: `clf__C`, `clf__gamma`
- Logistische Regression: `clf__C`

(Parameternamen im Grid brauchen das Pipeline-Praefix `clf__`, weil der Klassifikator
in der Pipeline unter dem Namen `"clf"` steckt.)

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: grid_1 = {...}  Parameter-Grid fuer dein 1. Top-Modell
# TODO: suche_1 = GridSearchCV(modelle["..."], grid_1, cv=cv, scoring="roc_auc", n_jobs=-1)
# TODO: suche_1.fit(X_train, y_train)

print(f"Bestes CV-ROC-AUC (Modell 1): {suche_1.best_score_:.4f}")
print(f"Beste Parameter: {suche_1.best_params_}")

In [ ]:
# TODO: dasselbe fuer dein 2. Top-Modell (grid_2, suche_2)

print(f"Bestes CV-ROC-AUC (Modell 2): {suche_2.best_score_:.4f}")
print(f"Beste Parameter: {suche_2.best_params_}")

## 6. Die einmalige Testauswertung

Jetzt erst kommt der Testsatz ins Spiel. **Aufgabe:** Waehle anhand von `best_score_`
das insgesamt beste der beiden getunten Modelle (`bestes_modell = suche_1.best_estimator_`
oder `suche_2.best_estimator_` — `GridSearchCV` hat es schon auf ganz Train refittet).
Werte einmal auf `X_test`/`y_test` aus: Confusion Matrix, Classification Report, ROC-Kurve + AUC.

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay, roc_auc_score)

# TODO: bestes_modell = ...
# TODO: y_pred = bestes_modell.predict(X_test)
# TODO: y_proba = bestes_modell.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))
print(f"Test-ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

fig, achsen = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["malignant", "benign"], ax=achsen[0])
RocCurveDisplay.from_predictions(y_test, y_proba, ax=achsen[1])
achsen[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.tight_layout()
plt.show()

# Mini-Check
print(roc_auc_score(y_test, y_proba) > 0.97)

## 7. Interpretation: Permutation Importance

**Aufgabe:** Berechne `permutation_importance` (aus `sklearn.inspection`) fuer dein
finales Modell **auf dem Testsatz** (Skript 3.1: das ist der ehrlichere Ort dafuer als
die Trainingsdaten). Plotte die Top 10 Features als horizontales Balkendiagramm.

In [ ]:
from sklearn.inspection import permutation_importance

# TODO: wichtigkeit = permutation_importance(bestes_modell, X_test, y_test,
#                                             n_repeats=30, random_state=42, scoring="roc_auc")

reihenfolge = np.argsort(wichtigkeit.importances_mean)[-10:]
plt.figure(figsize=(7, 5))
plt.barh(X.columns[reihenfolge], wichtigkeit.importances_mean[reihenfolge],
         xerr=wichtigkeit.importances_std[reihenfolge])
plt.xlabel("Abfall in ROC-AUC beim Verwuerfeln")
plt.title("Permutation Importance (Top 10, auf Testdaten)")
plt.tight_layout()
plt.show()

**Aufgabe:** Wirken die Top-Features medizinisch plausibel (z. B. `worst radius`,
`worst concave points`, `worst perimeter` — groessere/unregelmaessigere Zellkerne
deuten auf Malignitaet)? Kurze Notiz:

*(Deine Notiz hier ...)*

## 8. Lernkurve: Bias- oder Varianz-Problem?

**Aufgabe:** Plotte die Lernkurve (`learning_curve`, Skript 2.5) fuer dein finales
Modell: Trainings- und CV-Score gegen die Trainingsmenge (`train_sizes=np.linspace(0.1, 1.0, 8)`,
`scoring="roc_auc"`, `cv=cv`, auf `X_train`/`y_train`). Diagnostiziere: Lohnt sich mehr Daten?

In [ ]:
from sklearn.model_selection import learning_curve

# TODO: groessen, train_scores, val_scores = learning_curve(bestes_modell, X_train, y_train,
#            train_sizes=np.linspace(0.1, 1.0, 8), cv=cv, scoring="roc_auc", n_jobs=-1)

plt.figure(figsize=(7, 4.5))
plt.plot(groessen, train_scores.mean(axis=1), marker="o", label="Training")
plt.plot(groessen, val_scores.mean(axis=1), marker="s", label="Validierung (CV)")
plt.xlabel("Trainingsgroesse"); plt.ylabel("ROC-AUC"); plt.legend()
plt.title("Lernkurve des finalen Modells")
plt.tight_layout()
plt.show()

**Aufgabe:** Bias- oder Varianz-Problem (oder keins von beiden)? Wuerde mehr Daten helfen?

*(Deine Notiz hier ...)*

## Geschafft — was du jetzt kannst

- Mehrere Modellfamilien konsistent in Pipelines verpacken
- Modelle fair per stratifizierter Kreuzvalidierung vergleichen (Mittelwert **und** Streuung lesen)
- Hyperparameter systematisch per GridSearchCV tunen, ohne den Testsatz anzufassen
- Am Ende **einmal** ehrlich testen und das Ergebnis mit Metriken, Kurven und
  Permutation Importance einordnen — inklusive Lernkurven-Diagnose

**Reflexionsfragen (beantworte in 1-2 Saetzen je Frage):**

1. Warum waere es Selbstbetrug gewesen, `GridSearchCV` direkt mit `scoring="accuracy"`
   auf diesem Datensatz laufen zu lassen, ohne ueber Klassenbalance nachzudenken?
2. Angenommen, `SVC` haette in Schritt 4 die beste CV-ROC-AUC, aber die groesste
   Streuung ueber die 5 Folds gehabt. Wuerdest du es trotzdem waehlen? Wovon haengt das ab?
3. Was wuerde sich an Schritt 6 aendern, wenn Fehlalarme (False Positives) fuer diese
   Anwendung 10x teurer waeren als uebersehene Faelle (False Negatives)?

**Bonusaufgaben:**
1. Ersetze `GridSearchCV` durch `RandomizedSearchCV` mit einem groesseren Parameterraum
   (z. B. Verteilungen statt Listen) — vergleiche Laufzeit und Ergebnis.
2. Probiere `CalibratedClassifierCV` (Skript 3.2) auf deinem finalen Modell und
   vergleiche das Reliability Diagram vorher/nachher.